# Cholesky decomposition with jaxmg.potrs

Here, we give an example of calling `jaxmg.potrs`, which solves the linear system $Ax=b$ for symmetric, positive-definite $A$ via a Cholesky decomposition.

The input arrays use an ordinary 2D JAX mesh. JAXMg pads local shards when needed, enters one fused native FFI call, redistributes to cuSOLVERMp layout, solves, and returns the result in the JAX-facing layout.

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P, NamedSharding
from jaxmg import potrs
print(f"Devices: {jax.devices()}")
# Assumes we have at least one GPU available
devices = jax.devices("gpu")
N = 12
T_A = 3
dtype = jnp.float64
# Create diagonal matrix and `b` all equal to one
A = jnp.diag(jnp.arange(N, dtype=dtype) + 1)
b = jnp.ones((N, 1), dtype=dtype)
ndev = len(devices)
# Make a degenerate 2D mesh and place data
mesh = jax.make_mesh((ndev, 1), ("pr", "pc"))
sharding = NamedSharding(mesh, P("pr", "pc"))
A = jax.device_put(A, sharding)
b = jax.device_put(b, sharding)
# Call potrs
out = potrs(A, b, T_A=T_A)
print(out)
expected_out = 1.0 / (jnp.arange(N, dtype=dtype) + 1)
print(jnp.allclose(out.flatten(), expected_out))

mkdir -p failed for path /home/rwiersema/.cache/matplotlib: [Errno 13] Permission denied: '/home/rwiersema'
Matplotlib created a temporary cache directory at /tmp/matplotlib-kkpjcj85 because there was an issue with the default path (/home/rwiersema/.cache/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Devices: [CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2)]
[[1.        ]
 [0.5       ]
 [0.33333333]
 [0.25      ]
 [0.2       ]
 [0.16666667]
 [0.14285714]
 [0.125     ]
 [0.11111111]
 [0.1       ]
 [0.09090909]
 [0.08333333]]
True
